In [1]:
import torch
import os 
import json
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import CIFAR100
from torchvision import transforms, models
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# TIỀN XỬ LÝ DỮ LIỆU & AUGMENTATION
# ==========================================
# Chuẩn hóa theo Mean và Std của bộ dữ liệu CIFAR-100
transform_train = transforms.Compose([
    transforms.Resize(224),  # Phóng to ảnh thành 224x224 để phù hợp với ViT
    transforms.RandomCrop(224, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))
])

transform_test = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))
])
# 4. Khởi tạo Dataset chuẩn
raw_train_dataset  = CIFAR100(root="data", train=True, download=True, transform=transform_train)
raw_val_dataset    = CIFAR100(root="data", train=True, download=False, transform=transform_test)
final_test_dataset = CIFAR100(root="data", train=False, download=True, transform=transform_test)

# 5. Chia Train (90%) - Val (10%) từ 50,000 mẫu
total_samples = len(raw_train_dataset)
train_size = int(0.9 * total_samples)
val_size = total_samples - train_size

generator = torch.Generator().manual_seed(42)
permuted_indices = torch.randperm(total_samples, generator=generator).tolist()

train_indices = permuted_indices[:train_size]
val_indices   = permuted_indices[train_size:]

TRAIN_SUBSET = Subset(raw_train_dataset, train_indices)
VAL_SUBSET   = Subset(raw_val_dataset, val_indices)

# 6. DataLoader với num_workers=0 để tránh lỗi Shared Memory (shm)
BATCH_SIZE = 64
train_loader = DataLoader(TRAIN_SUBSET, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader   = DataLoader(VAL_SUBSET, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(final_test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

In [2]:
import timm
model = timm.create_model(
    'vit_base_patch16_224', 
    pretrained=True, 
    img_size=224, 
    num_classes=100
).to(device)

In [3]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Số lượng tham số cần huấn luyện (chỉ tính lớp head): {trainable_params:,}")

Số lượng tham số cần huấn luyện (chỉ tính lớp head): 85,875,556


In [4]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), 
    lr=5e-4, 
    weight_decay=1e-4
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

In [5]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)
        
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

In [6]:
def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)
            
    return running_loss / total, correct / total

In [7]:
def save_and_update_history_json(new_history, filepath="vit_cifar100_history_pretrain.json"):
    """Ghi hoặc cập nhật lịch sử huấn luyện vào file JSON."""
    history = {}
    if os.path.exists(filepath):
        with open(filepath, "r", encoding="utf-8") as f:
            try:
                history = json.load(f)
            except json.JSONDecodeError:
                history = {}

    # Nối tiếp danh sách lịch sử nếu đã có từ trước
    for key, values in new_history.items():
        if key in history:
            history[key].extend(values)
        else:
            history[key] = values

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(history, f, indent=4, ensure_ascii=False)

    return history

In [8]:
best_val_acc = 0.0
checkpoint_path = "best_vit_cifar100_pretrain.pth"
json_path = "vit_cifar100_history_pretrain.json"

current_run_history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}
BATCH_SIZE = 64
NUM_EPOCHS = 50
print("\n--- BẮT ĐẦU HUẤN LUYỆN ---")
for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc     = evaluate(model, val_loader, criterion, device)
    
    # Cập nhật Learning Rate Scheduler
    scheduler.step()
    
    # Cập nhật thông số từng epoch vào dict
    current_run_history['train_loss'].append(round(train_loss, 4))
    current_run_history['train_acc'].append(round(train_acc * 100, 2))
    current_run_history['val_loss'].append(round(val_loss, 4))
    current_run_history['val_acc'].append(round(val_acc * 100, 2))   
    
    # In định dạng chuỗi f-string
    print(f"Epoch [{epoch+1:02d}/{NUM_EPOCHS:02d}] | "
          f"Train Loss: {train_loss:.4f} - Train Acc: {train_acc*100:.2f}% | "
          f"Val Loss: {val_loss:.4f} - Val Acc: {val_acc*100:.2f}%")
    
    # Lưu checkpoint mô hình tốt nhất dựa trên Validation Accuracy
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), checkpoint_path)
        print(f"  --> Đã lưu model mới tốt nhất với Val Acc: {val_acc*100:.2f}%")

# LƯU & CẬP NHẬT VÀO FILE JSON
full_history = save_and_update_history_json(current_run_history, filepath=json_path)

# ==========================================
# 6. ĐÁNH GIÁ TRÊN TẬP TEST
# ==========================================
print("\n--- ĐÁNH GIÁ TRÊN TẬP TEST ---")
# Load lại trọng số tốt nhất đã lưu
model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f} | Test Accuracy: {test_acc*100:.2f}%")


--- BẮT ĐẦU HUẤN LUYỆN ---


OutOfMemoryError: CUDA out of memory. Tried to allocate 38.00 MiB. GPU 0 has a total capacity of 7.66 GiB of which 15.62 MiB is free. Including non-PyTorch memory, this process has 4.30 GiB memory in use. Of the allocated memory 4.05 GiB is allocated by PyTorch, and 91.43 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
import matplotlib.pyplot as plt

def plot_training_history(json_path="vit_cifar100_history_pretrain.json"):
    # 1. Kiểm tra sự tồn tại của file JSON
    if not os.path.exists(json_path):
        print(f"Lỗi: Không tìm thấy file '{json_path}'. Hãy đảm bảo file đã được tạo sau khi train.")
        return

    # 2. Đọc dữ liệu từ file JSON
    with open(json_path, "r", encoding="utf-8") as f:
        history = json.load(f)

    train_loss = history.get("train_loss", [])
    val_loss = history.get("val_loss", [])
    train_acc = history.get("train_acc", [])
    val_acc = history.get("val_acc", [])

    epochs = range(1, len(train_loss) + 1)

    # 3. Tạo đồ thị
    plt.figure(figsize=(14, 5))

    # --- Đồ thị 1: Train Loss vs Val Loss ---
    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_loss, label="Train Loss", color="#1f77b4", linewidth=2)
    plt.plot(epochs, val_loss, label="Val Loss", color="#ff7f0e", linewidth=2, linestyle="--")
    plt.title("Biểu đồ Loss qua các Epochs", fontsize=14, fontweight="bold")
    plt.xlabel("Epoch", fontsize=12)
    plt.ylabel("Loss", fontsize=12)
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.legend(fontsize=11)

    # --- Đồ thị 2: Train Accuracy vs Val Accuracy ---
    plt.subplot(1, 2, 2)
    plt.plot(epochs, train_acc, label="Train Accuracy (%)", color="#2ca02c", linewidth=2)
    plt.plot(epochs, val_acc, label="Val Accuracy (%)", color="#d62728", linewidth=2, linestyle="--")
    plt.title("Biểu đồ Accuracy qua các Epochs", fontsize=14, fontweight="bold")
    plt.xlabel("Epoch", fontsize=12)
    plt.ylabel("Accuracy (%)", fontsize=12)
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.legend(fontsize=11)

    # Tối ưu khoảng cách giữa các đồ thị và lưu/hiển thị
    plt.tight_layout()
    plt.savefig("vit_cifar100_learning_curves_pretrain.png", dpi=300)
    print("Đã lưu biểu đồ vào file 'vit_cifar100_learning_curves_pretrain.png'")
    plt.show()

plot_training_history("vit_cifar100_history_pretrain.json")